# PlayeRank Tutorial

## Import libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [2]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
import plotly.graph_objs as go
import plotly.offline as py
from pappalardo.playerank.features import (
    centerOfPerformanceFeature,
    goalScoredFeatures,
    matchPlayedFeatures,
    plainAggregation,
    playerankFeatures,
    qualityFeatures,
    relativeAggregation,
    roleFeatures,
)
from pappalardo.playerank.models import Clusterer, Rater, Weighter

from config import paths

In [3]:
# Define Wyscout data file paths
EVENTS_PATHS = str(paths.WYSCOUT_PAPPALARDO_DIR / "events" / "*.json")
MATCHES_PATHS = str(paths.WYSCOUT_PAPPALARDO_DIR / "matches" / "*.json")
PLAYERS_FILEPATH = str(paths.WYSCOUT_PAPPALARDO_DIR / "players.json")

# Define obtained features file paths
FEATURE_WEIGHTS_FILEPATH = str(Path().cwd() / "feature_weights.json")
ROLE_MATRIX_FILEPATH = str(Path().cwd() / "role_matrix.json")

## Feature Weights Computation

In [4]:
# Create quality features for teams
quality_features = qualityFeatures.qualityFeatures()
quality = quality_features.createFeature(
    events_path=EVENTS_PATHS,
    players_file=PLAYERS_FILEPATH,
    entity="team",
)

[qualityFeatures] added 643150 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_England.json
[qualityFeatures] added 78140 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_European_Championship.json
[qualityFeatures] added 632807 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_France.json
[qualityFeatures] added 519407 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Germany.json
[qualityFeatures] added 647372 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Italy.json
[qualityFeatures] added 628659 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Spain.json
[qualityFeatures] added 101759 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_World_Cup.json


In [5]:
# Create goal scored features for teams
goal_scored_features = goalScoredFeatures.goalScoredFeatures()
goals = goal_scored_features.createFeature(matches_path=MATCHES_PATHS)

[GoalScored features] added 380 matches
[GoalScored features] added 51 matches
[GoalScored features] added 380 matches
[GoalScored features] added 306 matches
[GoalScored features] added 380 matches
[GoalScored features] added 380 matches
[GoalScored features] added 64 matches


In [6]:
# Merge quality features and goals scored
aggregation = relativeAggregation.relativeAggregation()
aggregation.set_features([quality, goals])
df = aggregation.aggregate(to_dataframe=True)

[relativeAggregation] added 154945 features
[relativeAggregation] added 3882 features
[relativeAggregation] matches aggregated: 3882


In [7]:
# Fit Weighter model to compute feature weights
weighter = Weighter.Weighter(label_type="w-dl")
weighter.fit(df, target="Shot-Shot-accurate", var_threshold=0.02, filename=FEATURE_WEIGHTS_FILEPATH)
print(f"Features weights stored in {FEATURE_WEIGHTS_FILEPATH}")

[Weighter] filtered features:
('Others on the ball-Touch-assist', 0.0035933824218357044)
('Pass-Launch-key pass', 0.0010293349853655535)
('Pass-Hand pass-accurate', 0.01079262244454584)
('Goalkeeper leaving line-Goalkeeper leaving line', 0.0007722003110301022)
('Pass-Hand pass-not accurate', 0.0005149329220240262)
('Pass-Simple pass', 0.0005149329220240261)
Features weights stored in c:\Users\cristian\Desktop\uchile\soccer-kpis\models\playerank\feature_weights.json


In [8]:
# Load and plot feature weights
feature_data = json.load(open(FEATURE_WEIGHTS_FILEPATH))
fig = go.Figure(
    [
        go.Bar(
            x=list(feature_data.keys()),
            y=list(feature_data.values()),
        )
    ]
)
py.iplot(fig)

## Role Matrix Computation

In [9]:
# Create center of performance features
center_of_performance_features = centerOfPerformanceFeature.centerOfPerformanceFeature()
center_performance = center_of_performance_features.createFeature(
    events_path=EVENTS_PATHS,
    players_file=PLAYERS_FILEPATH,
)

[centerOfPerformanceFeature] added 643150 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_England.json
[centerOfPerformanceFeature] added 78140 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_European_Championship.json
[centerOfPerformanceFeature] added 632807 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_France.json
[centerOfPerformanceFeature] added 519407 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Germany.json
[centerOfPerformanceFeature] added 647372 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Italy.json
[centerOfPerformanceFeature] added 628659 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Spain.json
[centerOfPerformanceFeature] added 101759 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_World_Cup.j

In [10]:
# Realize plain aggregation to get a DataFrame
aggregation = plainAggregation.plainAggregation()
aggregation.set_features([center_performance])
df = aggregation.aggregate(to_dataframe=True)

[plainAggregation] added 139398 features
[plainAggregation] matches aggregated: 46466


In [11]:
# Cluster players based on their center of performance
clusterer = Clusterer.Clusterer(verbose=True, k_range=(8, 9))
clusterer.fit(
    player_ids=df["entity"],
    match_ids=df["match"],
    dataframe=df[["avg_x", "avg_y"]],
    kind="single",
)

matrix_role = clusterer.get_clusters_matrix(kind="single")
json.dump(matrix_role, open(ROLE_MATRIX_FILEPATH, "w"), indent=2)

FITTING kmeans...

n_clust	|silhouette
---------------------
8	|0.37
9	|0.3694
Best: n_clust=8 (silhouette=0.37)

DONE.


In [12]:
# Create a dictionary in the form: role -> [list of points associated to the role]
role_points = defaultdict(list)
for x in matrix_role:
    for y in matrix_role[x]:
        role = matrix_role[x][y]
        role_points[role].append((x, y))

# Plot the center of performance roles
traces = []
for role in role_points:
    traces.append(
        go.Scatter(
            x=[x[0] for x in role_points[role]],
            y=[x[1] for x in role_points[role]],
            mode="markers",
            opacity=0.8,
            marker={
                "size": 10,
                "line": {"width": 0.1, "color": "white"},
            },
            name=role,
        )
    )

# Plot soccer field with roles
fig = go.Figure(
    data=traces
    + [
        go.Scatter(
            showlegend=False,
            y=[0, 100],
            x=[50, 50],
            mode="lines",
            line={"width": 2, "color": "white"},
        ),
        go.Scatter(
            showlegend=False,
            y=[20, 20, 80, 80],
            x=[0, 16, 16, 0],
            mode="lines",
            line={"width": 2, "color": "white"},
        ),
        go.Scatter(
            showlegend=False,
            y=[20, 20, 80, 80],
            x=[100, 84, 84, 100],
            mode="lines",
            line={"width": 2, "color": "white"},
        ),
        go.Scatter(
            showlegend=False,
            y=[35, 35, 65, 65],
            x=[100, 94, 94, 100],
            mode="lines",
            line={"width": 2, "color": "white"},
        ),
        go.Scatter(
            showlegend=False,
            y=[35, 35, 65, 65],
            x=[0, 6, 6, 0],
            mode="lines",
            line={"width": 2, "color": "white"},
        ),
    ],
    layout=go.Layout(
        hovermode="closest",
        autosize=True,
        width=550,
        height=400,
        plot_bgcolor="rgb(59,205,55)",
        xaxis={
            "range": [0, 100],
            "showgrid": False,
            "showticklabels": False,
        },
        yaxis={
            "range": [0, 100],
            "showgrid": False,
            "showticklabels": False,
        },
        shapes=[
            {
                "type": "circle",
                "xref": "x",
                "yref": "y",
                "y0": 30,
                "x0": 35,
                "y1": 70,
                "x1": 65,
                "line": {"color": "white"},
            }
        ],
    ),
)

py.iplot(fig)

## PlayeRank Scores Computation

In [13]:
# Create quality features for players
quality_features = qualityFeatures.qualityFeatures()
quality = quality_features.createFeature(
    events_path=EVENTS_PATHS,
    players_file=PLAYERS_FILEPATH,
    entity="player",
)

[qualityFeatures] added 643150 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_England.json
[qualityFeatures] added 78140 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_European_Championship.json
[qualityFeatures] added 632807 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_France.json
[qualityFeatures] added 519407 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Germany.json
[qualityFeatures] added 647372 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Italy.json
[qualityFeatures] added 628659 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Spain.json
[qualityFeatures] added 101759 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_World_Cup.json


In [14]:
# Create PlayeRank features using quality features
playerank_features = playerankFeatures.playerankFeatures()
playerank_features.set_features([quality])
playerank_scores = playerank_features.createFeature(FEATURE_WEIGHTS_FILEPATH)

[playerankFeatures] playerank scores computed. 51587 performance processed


In [15]:
# Create match played features
match_played_features = matchPlayedFeatures.matchPlayedFeatures()
match_played = match_played_features.createFeature(
    matches_path=MATCHES_PATHS,
    players_file=PLAYERS_FILEPATH,
)

[matchPlayedFeatures] processing 1941 matches
[matchPlayedFeatures] matches features computed. 261875 features processed


In [16]:
# Create center of performance features
center_of_performance_features = centerOfPerformanceFeature.centerOfPerformanceFeature()
center_performance = center_of_performance_features.createFeature(
    events_path=EVENTS_PATHS,
    players_file=PLAYERS_FILEPATH,
)

[centerOfPerformanceFeature] added 643150 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_England.json
[centerOfPerformanceFeature] added 78140 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_European_Championship.json
[centerOfPerformanceFeature] added 632807 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_France.json
[centerOfPerformanceFeature] added 519407 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Germany.json
[centerOfPerformanceFeature] added 647372 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Italy.json
[centerOfPerformanceFeature] added 628659 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_Spain.json
[centerOfPerformanceFeature] added 101759 events from C:\Users\cristian\Desktop\uchile\soccer-kpis\data\pappalardo\events\events_World_Cup.j

In [17]:
# Create role features
role_features = roleFeatures.roleFeatures()
role_features.set_features([center_performance])
roles = role_features.createFeature(matrix_role_file=ROLE_MATRIX_FILEPATH)

In [18]:
# Realize plain aggregation to get a DataFrame
aggregation = plainAggregation.plainAggregation()
aggregation.set_features([match_played, playerank_scores, roles])
df = aggregation.aggregate(to_dataframe=True)

[plainAggregation] added 261875 features
[plainAggregation] added 51587 features
[plainAggregation] added 46466 features
[plainAggregation] matches aggregated: 67292


In [19]:
# Compute final ratings using Rater model
rater = Rater.Rater(alpha_goal=0.1)  # Set goal weight as 10% of total performance score
df["ratings"] = rater.predict(
    df,
    score_feature="playerankScore",
    goal_feature="goalScored",
)
print(df.sort_values("ratings", ascending=False).head())

         match  entity  minutesPlayed  goalScored            timestamp  team  \
63971  2565577    3359           90.0         3.0  2017-09-09 18:45:00   676   
2533   2500023  120353           90.0         4.0  2018-03-17 17:30:00  1612   
21426  2500862   40810           90.0         2.0  2017-12-16 16:00:00  3767   
42148  2576216   69513           90.0         2.0  2018-02-25 11:30:00  3197   
17588  2500968   25410           90.0         4.0  2018-03-11 14:00:00  3775   

       playerankScore  roleCluster   ratings  
63971        0.215472          4.0  1.000000  
2533         0.053521          4.0  0.929725  
21426        0.259117          7.0  0.906743  
42148        0.237814          6.0  0.877297  
17588        0.004501          6.0  0.861966  


In [20]:
# Group ratings by player and roleCluster
grouped = df[["entity", "roleCluster", "ratings"]].groupby(["entity", "roleCluster"]).agg(["mean", "count"])
grouped.columns = ["ratings_mean", "ratings_count"]
grouped = grouped.reset_index()

# Load players data to map wyId and shortName
players = pd.read_json(PLAYERS_FILEPATH)
players = players[["wyId", "shortName"]]
players["entity"] = players["wyId"]

# Merge grouped ratings with players data
grouped = pd.merge(grouped, players, on="entity")
grouped["shortName"] = grouped["shortName"].apply(lambda x: x.encode().decode("unicode_escape"))

# Filter players with more than 15 ratings
grouped = grouped[grouped["ratings_count"] > 15]

In [21]:
colors = ["blue", "green", "red", "cyan", "magenta", "maroon", "crimson", "orange"]

cluster2color = {cluster: color for cluster, color in zip(grouped["roleCluster"].unique(), colors)}

traces = []
for cluster in grouped["roleCluster"].unique():
    trace = go.Box(
        y=grouped[grouped["roleCluster"] == cluster]["ratings_mean"].values,
        boxpoints="all",
        jitter=0.8,
        pointpos=0,
        name=cluster,
        text=grouped[grouped["roleCluster"] == cluster]["shortName"],
        marker=dict(color=cluster2color[cluster], size=3),
        hoverinfo="text",
    )
    traces.append(trace)

figure = go.Figure(
    data=traces,
    layout=go.Layout(
        title="Best performance of players by role",
        xaxis=dict(
            title="role",
            zeroline=True,
            showgrid=False,
            zerolinewidth=2,
        ),
        yaxis=dict(
            title="PlayeRank score",
            zeroline=True,
            showgrid=False,
        ),
        showlegend=False,
        # autosize=True,
    ),
)

py.iplot(figure)